# Datamine TREND Process Example

This notebook demonstrates how to configure and run the **`trend`** process wrapper in `dmstudio`.

## Process Description

## TREND

Fit a polynomial trend surface to a set of data.

A polynomial of order 1, 2 or 3 may be fitted. The coefficients are calculated, displayed in the Command window, and stored in the output file. A summary of the goodness of fit parameters are also displayed in the Command window.

* **Note** (If any record has any of *X, *Y or *Z values absent, the record is ignored and is not included in the record count.):

### Input Files:

* **in** (Undefined):
  Input data file. Must contain the fields X , Y , and VALUE.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing the surface coefficients. The fields are C0, CX, CY, CXY, CX2,

## CY2, CX2Y, CXY2, CX3, CY3.

  Required=Yes

### Fields:

* **x** (Numeric : IN):
  X Co-ordinate fieldname.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Y Co-ordinate fieldname.
  Default=Undefined
  Required=Yes

* **value** (Numeric : IN):
  Field to be fitted.
  Default=Undefined
  Required=Yes

### Parameters:

* **order**:
  Order of surface (1,2, or 3).
  Range=1,3
  Values=1,2,3
  Default=1
  Required=Yes

* **print**:
  >0, displays original samples, fitted points and differences (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **select**:
  Allows the user to select every nth record, where n=SELECT (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('trend')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute trend
print("Running trend...")
dm_cmd.trend(
    in_i='t_assays',  # required
    out_o='t_trend_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    value_f='optional',  # required
    order_p='required_val',  # required
    # print_p=0,  # optional
    # select_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("trend execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_trend_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")